# Failure-Mode Panel Notebook Template

Use this notebook to build a reusable supplementary failure-mode panel for a medical-AI manuscript. Replace placeholder column names with study-specific names.

## Expected input schema

| Column | Meaning |
|--------|---------|
| `case_id` | Unique case identifier |
| `unit_id` | Unit of analysis identifier |
| `y_true` | Reference-standard label |
| `y_score` | Model probability or score |
| `y_pred` | Thresholded model output |
| `group` | Clinically meaningful stratum |
| `operator_decision` | Clinician or operator decision, if available |
| `image_path` | Optional image or signal path for representative cases |
| `narrative` | One-line clinical narrative |


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_auc_score

DATA_PATH = Path("failure_mode_input.csv")
OUTPUT_DIR = Path("failure_mode_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
required = {"case_id", "unit_id", "y_true", "y_score", "y_pred", "group"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

summary = df.groupby("group").agg(
    n=("unit_id", "count"),
    events=("y_true", "sum"),
)
summary["event_rate"] = summary["events"] / summary["n"]
summary

## Panel A. Confusion matrix by clinically meaningful stratum

In [ ]:
rows = []
for group, group_df in df.groupby("group"):
    tn, fp, fn, tp = confusion_matrix(group_df["y_true"], group_df["y_pred"], labels=[0, 1]).ravel()
    rows.append({
        "group": group,
        "true_positive": tp,
        "false_positive": fp,
        "true_negative": tn,
        "false_negative": fn,
        "error_rate": (fp + fn) / len(group_df),
    })

    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay.from_predictions(group_df["y_true"], group_df["y_pred"], ax=ax, colorbar=False)
    ax.set_title(str(group))
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"confusion_matrix_{group}.png", dpi=300)

error_table = pd.DataFrame(rows)
error_table

## Panel B. Calibration

In [ ]:
prob_true, prob_pred = calibration_curve(df["y_true"], df["y_score"], n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(prob_pred, prob_true, marker="o", label="Model")
ax.plot([0, 1], [0, 1], linestyle="--", color="black", label="Perfect calibration")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Observed frequency")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "calibration.png", dpi=300)

## Panel C. Subgroup performance

In [ ]:
performance_rows = []
for group, group_df in df.groupby("group"):
    if group_df["y_true"].nunique() < 2:
        auroc = None
    else:
        auroc = roc_auc_score(group_df["y_true"], group_df["y_score"])
    performance_rows.append({"group": group, "n": len(group_df), "events": int(group_df["y_true"].sum()), "auroc": auroc})

performance_table = pd.DataFrame(performance_rows)
performance_table

## Panel D. Representative false-positive and false-negative cases

In [ ]:
false_positive = df[(df["y_true"] == 0) & (df["y_pred"] == 1)].sort_values("y_score", ascending=False).head(6)
false_negative = df[(df["y_true"] == 1) & (df["y_pred"] == 0)].sort_values("y_score", ascending=True).head(6)
case_table = pd.concat([false_positive.assign(error_type="False positive"), false_negative.assign(error_type="False negative")])
case_columns = [col for col in ["case_id", "unit_id", "error_type", "y_score", "group", "operator_decision", "narrative", "image_path"] if col in case_table.columns]
case_table[case_columns].to_csv(OUTPUT_DIR / "representative_error_cases.csv", index=False)
case_table[case_columns]

## Panel E. Operator-versus-model concordance

In [ ]:
if "operator_decision" in df.columns:
    concordance = pd.crosstab(df["operator_decision"], df["y_pred"], margins=True)
    concordance.to_csv(OUTPUT_DIR / "operator_model_concordance.csv")
    display(concordance)
else:
    print("operator_decision column not available")